# A2.5 · Delegation that survives audit

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

---

**Risk.** On-behalf-of implemented as impersonation — the chain is unprovable afterwards.

**Control.** RFC 8693 token exchange with the `act` claim as a real delegation chain.

**This lab.** A three-hop delegation chain that survives audit.

| | |
|---|---|
| Open-source tooling | Keycloak, RFC 8693 |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A2.5"))

This is the flagship identity lab. A three-hop delegation chain that survives audit — and the four ways it usually doesn't.

**1 — the chain.** Each hop narrows, and each hop is recorded.

In [ ]:
from cybercommons import identity

alice = identity.mint("alice")
rev   = identity.exchange(alice, "reviewer-agent", {"repo:read"})
patch = identity.exchange(alice, "patch-agent",    {"repo:read", "repo:write"})
dep   = identity.exchange(patch, "deploy-agent",   {"repo:read"})

for t in (alice, rev, patch, dep):
    print(t.describe())

**2 — widening is refused, twice over.** Once against the token that was presented, once against the actor's own ceiling.

In [ ]:
for presented, actor, want, why in [
    (patch, "deploy-agent",   {"deploy:prod"},  "not in the presented token"),
    (alice, "reviewer-agent", {"secrets:read"}, "above the actor's ceiling"),
]:
    try:
        identity.exchange(presented, actor, want)
        print(f"GRANTED — this should not happen ({why})")
    except identity.DelegationError as e:
        print(f"refused ({why}):\n    {e}")

**3 — the anti-pattern.** Impersonation produces a token that works perfectly and destroys the audit trail.

In [ ]:
bad = identity.impersonate("alice", "patch-agent", {"repo:write"})
print("delegated    :", " → ".join(patch.chain()))
print("impersonated :", " → ".join(bad.chain()), "  ← the agent is invisible")

**4 — revocation is per-actor.** One identity dies; the rest live.

In [ ]:
reg = identity.Registry()
for t in (alice, rev, patch, dep):
    reg.record(t)
print(f"revoking reviewer-agent → {reg.revoke('reviewer-agent')} token(s) hit\n")
for t in (rev, patch, dep):
    ok, why = reg.valid(t)
    print(f"  {' → '.join(t.chain()):46s} valid={str(ok):5s} {why}")

### Expect

Four tokens print with strictly narrowing scopes and readable chains (`alice → patch-agent → deploy-agent`). Both widening attempts raise `DelegationError` naming which rule refused. The impersonated token's chain contains only `alice`. Revoking `reviewer-agent` invalidates one token and leaves the patch and deploy chains valid.

### Your turn

Run the same four scenarios against real Keycloak with RFC 8693 token exchange (`labs/a2-delegation` has the compose file). The properties should hold identically — if they don't, your realm configuration is the finding.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A2.5.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*